<a href="https://colab.research.google.com/github/Monicababu21/VoxTranscribe/blob/main/VoxTranscribe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install openai-whisper gradio nltk pandas
!apt-get -qq update
!apt-get -qq install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 5.3 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
import whisper
import gradio as gr
import nltk
import pandas as pd
import re
import os
import tempfile
from collections import Counter

# Download NLTK resources
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)

from nltk.corpus import stopwords

# Load Whisper model
print("Loading Whisper model...")

model = whisper.load_model("base")

print("Whisper model loaded successfully!")

Loading Whisper model...


100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 90.0MiB/s]


Whisper model loaded successfully!


In [3]:
def analyze_transcript(text):
    if not text or not text.strip():
        return {
            "words": 0,
            "characters": 0,
            "sentences": 0,
            "paragraphs": 0,
            "avg_word_length": 0,
            "estimated_reading_time": 0
        }

    words = re.findall(r"\b[\w']+\b", text)

    sentences = [
        s.strip()
        for s in re.split(r"[.!?]+", text)
        if s.strip()
    ]

    paragraphs = [
        p.strip()
        for p in re.split(r"\n\s*\n", text)
        if p.strip()
    ]

    avg_word_length = (
        sum(len(word) for word in words) / len(words)
        if words else 0
    )

    # Approximate reading time: 200 words/minute
    reading_time = len(words) / 200

    return {
        "words": len(words),
        "characters": len(text),
        "sentences": len(sentences),
        "paragraphs": max(1, len(paragraphs)) if text.strip() else 0,
        "avg_word_length": round(avg_word_length, 2),
        "estimated_reading_time": round(reading_time, 2)
    }


def extract_keywords(text, limit=10):
    if not text or not text.strip():
        return []

    words = re.findall(r"\b[a-zA-Z]{3,}\b", text.lower())

    stop_words = set(stopwords.words("english"))

    filtered_words = [
        word
        for word in words
        if word not in stop_words
    ]

    frequency = Counter(filtered_words)

    return frequency.most_common(limit)

In [4]:
def transcribe_audio(audio_file, language):
    if audio_file is None:
        return (
            "⚠️ Please upload or record an audio file.",
            "",
            "",
            None
        )

    try:
        print("Transcribing audio...")

        # Convert interface language selection
        language_map = {
            "English": "en",
            "Tamil": "ta",
            "Hindi": "hi",
            "Auto Detect": None
        }

        selected_language = language_map[language]

        result = model.transcribe(
            audio_file,
            language=selected_language,
            fp16=False
        )

        transcript = result["text"].strip()

        if not transcript:
            return (
                "⚠️ No speech could be detected.",
                "",
                "",
                None
            )

        # Analysis
        stats = analyze_transcript(transcript)
        keywords = extract_keywords(transcript)

        keyword_text = ", ".join(
            [f"{word} ({count})" for word, count in keywords]
        )

        analysis = f"""
### 📊 Transcript Analysis

| Metric | Value |
|---|---:|
| Words | {stats['words']} |
| Characters | {stats['characters']} |
| Sentences | {stats['sentences']} |
| Paragraphs | {stats['paragraphs']} |
| Average Word Length | {stats['avg_word_length']} characters |
| Estimated Reading Time | {stats['estimated_reading_time']} min |

### 🔑 Keywords

{keyword_text if keyword_text else "No significant keywords found."}
"""

        # Save transcript
        output_file = tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".txt",
            mode="w",
            encoding="utf-8"
        )

        output_file.write(transcript)
        output_file.close()

        return (
            transcript,
            analysis,
            f"Detected language: {result.get('language', 'Unknown')}",
            output_file.name
        )

    except Exception as e:
        return (
            f"❌ Error during transcription: {str(e)}",
            "",
            "",
            None
        )

In [5]:
def get_audio_info(audio_file):
    if audio_file is None:
        return "No audio selected."

    try:
        import wave

        if audio_file.lower().endswith(".wav"):
            with wave.open(audio_file, "rb") as audio:
                duration = audio.getnframes() / float(audio.getframerate())

            minutes = int(duration // 60)
            seconds = int(duration % 60)

            return f"🎧 Audio Duration: {minutes} min {seconds} sec"

        return "🎧 Audio file uploaded successfully."

    except:
        return "🎧 Audio file uploaded successfully."

In [7]:
# ============================================================
# CELL 6 — VOXTRANSCRIBE GRADIO UI
# ============================================================

import gradio as gr

# ------------------------------------------------------------
# Text analysis function for the Text Analysis tab
# ------------------------------------------------------------
def analyze_only(text):

    if not text or not text.strip():
        return "⚠️ Please enter some text to analyze."

    stats = analyze_transcript(text)
    keywords = extract_keywords(text)

    keyword_text = ", ".join(
        [f"{word} ({count})" for word, count in keywords]
    )

    return f"""
## 📊 Text Analysis

| Metric | Value |
|---|---:|
| Words | {stats['words']} |
| Characters | {stats['characters']} |
| Sentences | {stats['sentences']} |
| Paragraphs | {stats['paragraphs']} |
| Average Word Length | {stats['avg_word_length']} characters |
| Estimated Reading Time | {stats['estimated_reading_time']} min |

### 🔑 Keywords

{keyword_text if keyword_text else "No significant keywords found."}
"""


# ------------------------------------------------------------
# Function used by Speech-to-Text tab
# ------------------------------------------------------------
def transcribe_for_ui(audio_file, language):

    transcript, analysis, detected, file_path = transcribe_audio(
        audio_file,
        language
    )

    return transcript, analysis, detected, file_path


# ------------------------------------------------------------
# Custom CSS
# ------------------------------------------------------------
custom_css = """

body {
    background: #ffffff;
}

.gradio-container {
    max-width: 1200px !important;
    margin: auto;
}

h1 {
    color: #6d28d9;
}

button.primary {
    background: #7c3aed !important;
    border: none !important;
}

button.primary:hover {
    background: #6d28d9 !important;
}

textarea {
    border-radius: 12px !important;
}

"""


# ------------------------------------------------------------
# Main Application
# ------------------------------------------------------------
with gr.Blocks(
    title="VoxTranscribe — ASR Studio"
) as app:

    # Header
    gr.Markdown(
        """
        # 🎙️ VoxTranscribe
        ### Automatic Speech Recognition & Text Analysis Studio

        **Convert speech into text, analyze transcripts, and extract keywords.**
        """
    )

    with gr.Tabs():

        # ====================================================
        # TAB 1 — SPEECH TO TEXT
        # ====================================================
        with gr.Tab("🎙️ Speech to Text"):

            gr.Markdown(
                """
                ## Convert Speech to Text

                Upload an audio file or record your voice directly.
                """
            )

            audio_input = gr.Audio(
                sources=["upload", "microphone"],
                type="filepath",
                label="🎤 Upload or Record Audio"
            )

            language = gr.Dropdown(
                choices=[
                    "Auto Detect",
                    "English",
                    "Tamil",
                    "Hindi"
                ],
                value="Auto Detect",
                label="🌐 Select Language"
            )

            transcribe_button = gr.Button(
                "🎙️ Transcribe Speech",
                variant="primary"
            )

            transcript_output = gr.Textbox(
                label="📝 Transcript",
                placeholder="Your transcribed speech will appear here...",
                lines=10
            )

            detected_language = gr.Markdown()

            analysis_output = gr.Markdown()

            download_file = gr.File(
                label="📥 Download Transcript"
            )

            transcribe_button.click(
                fn=transcribe_for_ui,
                inputs=[
                    audio_input,
                    language
                ],
                outputs=[
                    transcript_output,
                    analysis_output,
                    detected_language,
                    download_file
                ]
            )

        # ====================================================
        # TAB 2 — TEXT ANALYSIS
        # ====================================================
        with gr.Tab("📊 Text Analysis"):

            gr.Markdown(
                """
                ## Analyze Text

                Enter any text to calculate statistics and extract keywords.
                """
            )

            analysis_text = gr.Textbox(
                label="📝 Enter Text",
                placeholder="Paste your text here...",
                lines=12
            )

            analyze_button = gr.Button(
                "📊 Analyze Text",
                variant="primary"
            )

            analysis_output_tab = gr.Markdown()

            analyze_button.click(
                fn=analyze_only,
                inputs=analysis_text,
                outputs=analysis_output_tab
            )

        # ====================================================
        # TAB 3 — ABOUT
        # ====================================================
        with gr.Tab("ℹ️ About"):

            gr.Markdown(
                """
                # 🎙️ About VoxTranscribe

                **VoxTranscribe** is an Automatic Speech Recognition
                and Text Analysis project.

                ### 🚀 Features

                - 🎤 Upload audio
                - 🎙️ Record audio using microphone
                - 📝 Speech-to-text conversion
                - 🌐 English, Tamil and Hindi support
                - 🔍 Automatic language detection
                - 📊 Transcript statistics
                - 🔑 Keyword extraction
                - 📥 Transcript download
                - 📖 Text analysis

                ### 🛠️ Technologies

                - Python
                - OpenAI Whisper
                - Gradio
                - NLTK
                - Pandas
                - FFmpeg

                ### 🎓 Project Type

                **Text Analysis & Processing — Automatic Speech Recognition**
                """
            )


# ============================================================
# LAUNCH
# ============================================================

app.launch(
    share=True,
    theme=gr.themes.Soft(
        primary_hue="violet",
        secondary_hue="purple"
    ),
    css=custom_css
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://82a139c464885acfbe.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
